In [81]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

In [83]:
#Parameters
CSV_FILE      = "first_25000_rows.csv"   # path to your dataset
TIME_INTERVAL = "1ns"                  # e.g. '5S', '30S', '1Min', '15Min' - club events in the time interval - not used in the implementation
BOOK_DEPTH    = 10                      # 00–09 columns present in file


In [85]:
# Event-level OFI for ONE stock 
def compute_event_ofi_one_stock(df_sym: pd.DataFrame,
                                depth: int = BOOK_DEPTH) -> pd.DataFrame:

    out = pd.DataFrame(index=df_sym.index)

    def bid_delta(p, p_prev, q, q_prev):
        return np.where(np.isnan(p) & np.isnan(p_prev), 0,
               np.where(np.isnan(p),            -q_prev,
               np.where(np.isnan(p_prev),        q,
               np.where(p > p_prev,              q,
               np.where(p == p_prev,             q - q_prev,
                                                   -q_prev)))))

    def ask_delta(p, p_prev, q, q_prev):
        return np.where(np.isnan(p) & np.isnan(p_prev), 0,
               np.where(np.isnan(p),             q_prev,
               np.where(np.isnan(p_prev),       -q,
               np.where(p > p_prev,             -q_prev,     
               np.where(p == p_prev,             q_prev - q,
                                                  q)))))      

    for m in range(depth):
        bp   = df_sym[f"bid_px_{m:02d}"]
        bpp  = bp.shift(1)
        bs   = df_sym[f"bid_sz_{m:02d}"]
        bsp  = bs.shift(1)

        ap   = df_sym[f"ask_px_{m:02d}"]
        app  = ap.shift(1)
        a_sz = df_sym[f"ask_sz_{m:02d}"]
        a_sz_prev = a_sz.shift(1)

        I_b = bid_delta(bp, bpp, bs, bsp)
        I_a = ask_delta(ap, app, a_sz, a_sz_prev)

        out[f"OFI_L{m+1}"] = I_b + I_a

    return out.fillna(0)   # drop first row (needs prev state)

In [87]:
#Load data & build event-level OFI for ALL stocks
print("Loading CSV …")
raw = pd.read_csv(CSV_FILE, parse_dates=["ts_event"])
raw = raw.sort_values(["symbol", "ts_event"]).reset_index(drop=True)

event_ofi_parts = []
for sym, grp in raw.groupby("symbol", sort=False):
    part = compute_event_ofi_one_stock(grp)
    part["symbol"]   = sym
    part["ts_event"] = grp.loc[part.index, "ts_event"].values
    event_ofi_parts.append(part)

event_ofi = pd.concat(event_ofi_parts, ignore_index=True)

Loading CSV …


In [102]:
#Aggregate into user-chosen time buckets
event_ofi["time_bin"] = event_ofi["ts_event"].dt.floor(TIME_INTERVAL)
ofi_cols = [f"OFI_L{i}" for i in range(1, BOOK_DEPTH + 1)]

"""------To be used for aggregation in buckets ------
        agg = (event_ofi
       .groupby(["time_bin", "symbol"])[ofi_cols]
       .sum()
       .reset_index())
       ----------------------------------------------"""

agg = event_ofi.copy()

# Best-level scalar and optional multi-level sum
agg["Best_Level_OFI"]  = agg["OFI_L1"]
agg["Multi_Level_Sum"] = agg[ofi_cols].sum(axis=1)

In [91]:
# Integrated OFI
print("Running PCA for Integrated OFI …")
integrated_vals = []

for sym, grp in agg.groupby("symbol", sort=False):
    mat = grp[ofi_cols].values
    if len(mat) < 2:
        w = np.ones(BOOK_DEPTH) / BOOK_DEPTH        # fallback weights
    else:
        pca = PCA(n_components=1)
        pca.fit(mat)
        w = pca.components_[0]
        w /= np.abs(w).sum()                         # L1-normalise
    integrated_vals.append(pd.Series(mat @ w, index=grp.index))

agg["Integrated_OFI"] = pd.concat(integrated_vals).sort_index()

Running PCA for Integrated OFI …


In [96]:
#Cross-Asset OFI 

"""------To be used if using aggregation in buckets (comment the uncommented code)------
pivot_int = agg.pivot(index="time_bin", columns="symbol",
                      values="Integrated_OFI")
cross = pivot_int.apply(lambda r: r.sum() - r, axis=1)   # row-wise
cross_long = cross.stack().rename("Cross_Asset_OFI").reset_index()
agg = agg.merge(cross_long, on=["time_bin", "symbol"])
-------------------------------------------------------"""

bin_totals = agg.groupby("time_bin")["Integrated_OFI"] \
                .sum() \
                .rename("Bin_Total_OFI") \
                .reset_index()

# 2) Merge back and subtract own value
agg = agg.merge(bin_totals, on="time_bin", how="left")
agg["Cross_Asset_OFI"] = agg["Bin_Total_OFI"] - agg["Integrated_OFI"]

# 3) Drop the helper column if you like
agg.drop(columns="Bin_Total_OFI", inplace=True)
# ─────────────────────────────────────────────────

In [98]:
# DataFrame

final_cols = ["time_bin", "symbol",
              "Best_Level_OFI", "Multi_Level_Sum",
              "Integrated_OFI", "Cross_Asset_OFI"] + ofi_cols

df_final = agg[final_cols].sort_values(["time_bin", "symbol"]) \
                          .reset_index(drop=True)

pd.set_option("display.expand_frame_repr", False)  # print wide
print("\n==== FIRST 10 ROWS ====")
print(df_final.head(10))
print("\n==== LAST 10 ROWS ====")
print(df_final.tail(10))

#save final csv
df_final.to_csv("ofi_features.csv", index=False)


==== FIRST 10 ROWS ====
                       time_bin symbol  Best_Level_OFI  Multi_Level_Sum  Integrated_OFI  Cross_Asset_OFI  OFI_L1  OFI_L2  OFI_L3  OFI_L4  OFI_L5  OFI_L6  OFI_L7  OFI_L8  OFI_L9  OFI_L10
0 2024-10-21 11:54:29.221064336   AAPL           -61.0           -435.0      -34.592638              0.0   -61.0     9.0   192.0  -122.0   -14.0    85.0  -190.0    66.0   -55.0   -345.0
1 2024-10-21 11:54:29.223769812   AAPL             2.0              2.0        0.035380              0.0     2.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0      0.0
2 2024-10-21 11:54:29.225030400   AAPL             3.0              3.0        0.053070              0.0     3.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0      0.0
3 2024-10-21 11:54:29.712434212   AAPL             0.0            200.0       20.961596              0.0     0.0     0.0   200.0     0.0     0.0     0.0     0.0     0.0     0.0      0.0
4 2024-10-21 11:54:29.764673165   AAPL       